# 🏔️ Nepal Flood & Landslide Predictor — Model Training
**Trains a classifier to predict flood/landslide risk from weather data.**

Pipeline:
1. Load & explore `Nepal_Disaster_Final.csv`
2. Feature engineering
3. Handle class imbalance
4. Train Logistic Regression + SVM
5. Evaluate & compare both models
6. Save best model for Streamlit app

## Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils import resample

print('✅ All libraries loaded!')

## Cell 2 — Load Dataset & Quick Exploration

In [ ]:
df = pd.read_csv('Nepal_Disaster_Final.csv')
df['date'] = pd.to_datetime(df['date'])

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
print('Event distribution:')
print(df['event_type'].value_counts())
print()
df.head()

## Cell 3 — Feature Engineering
We extract time-based features from the date column and encode district as a number.

In [ ]:
df_model = df.copy()

# ── Time features ─────────────────────────────────────────────────────────────
df_model['month']      = df_model['date'].dt.month
df_model['day_of_year']= df_model['date'].dt.dayofyear

# Nepal monsoon season flag (June=6 to September=9)
df_model['is_monsoon'] = df_model['month'].apply(lambda m: 1 if 6 <= m <= 9 else 0)

# Season encoding
def get_season(month):
    if month in [12, 1, 2]:  return 0  # Winter
    elif month in [3, 4, 5]: return 1  # Spring/Pre-monsoon
    elif month in [6, 7, 8, 9]: return 2  # Monsoon
    else: return 3  # Post-monsoon

df_model['season'] = df_model['month'].apply(get_season)

# ── District encoding ─────────────────────────────────────────────────────────
le = LabelEncoder()
df_model['district_encoded'] = le.fit_transform(df_model['district'])

# Save the label encoder — needed in Streamlit app later
joblib.dump(le, 'district_label_encoder.pkl')
print('✅ District label encoder saved!')

# ── Target encoding ───────────────────────────────────────────────────────────
# 0 = No Event, 1 = Flood, 2 = Landslide
event_map = {'No Event': 0, 'Flood': 1, 'Landslide': 2}
df_model['target'] = df_model['event_type'].map(event_map)

print('Feature engineering complete!')
print()
print('New columns added: month, day_of_year, is_monsoon, season, district_encoded, target')
print()
print('Target distribution:')
print(df_model['target'].value_counts().rename({0:'No Event', 1:'Flood', 2:'Landslide'}))
df_model[['date','district','month','is_monsoon','season','precipitation',
           'rainfall_3day_sum','target']].head(10)

## Cell 4 — Correlation Heatmap
Understand which features matter most.

In [ ]:
feature_cols = ['precipitation', 'rainfall_3day_sum', 'rainfall_7day_sum',
                'rainfall_30day_sum', 'T2M', 'temp_7day_avg',
                'month', 'is_monsoon', 'season', 'district_encoded', 'target']

corr = df_model[feature_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()
print('📊 Heatmap saved!')

## Cell 5 — Handle Class Imbalance
We have 169k No Event rows vs ~20k each for Flood/Landslide.
We'll downsample No Event to balance the classes — same idea as stratify=Y you used before.

In [ ]:
# Separate classes
df_no_event  = df_model[df_model['target'] == 0]
df_flood     = df_model[df_model['target'] == 1]
df_landslide = df_model[df_model['target'] == 2]

print(f'Before balancing:')
print(f'  No Event:  {len(df_no_event):,}')
print(f'  Flood:     {len(df_flood):,}')
print(f'  Landslide: {len(df_landslide):,}')

# Downsample No Event to 2x the average of flood/landslide
target_size = int((len(df_flood) + len(df_landslide)) / 2 * 2)

df_no_event_down = resample(df_no_event,
                            replace=False,
                            n_samples=target_size,
                            random_state=42)

df_balanced = pd.concat([df_no_event_down, df_flood, df_landslide])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nAfter balancing:')
print(df_balanced['target'].value_counts().rename({0:'No Event', 1:'Flood', 2:'Landslide'}))
print(f'\nTotal training rows: {len(df_balanced):,}')

## Cell 6 — Prepare Features & Split Data

In [ ]:
FEATURES = [
    'precipitation',
    'rainfall_3day_sum',
    'rainfall_7day_sum',
    'rainfall_30day_sum',
    'T2M',
    'temp_7day_avg',
    'month',
    'is_monsoon',
    'season',
    'district_encoded'
]

X = df_balanced[FEATURES]
y = df_balanced['target']

# Train/test split — stratified to keep class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — same StandardScaler you used in Diabetes project
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler for Streamlit app
joblib.dump(scaler, 'feature_scaler.pkl')

print(f'Training set: {X_train_scaled.shape}')
print(f'Test set:     {X_test_scaled.shape}')
print(f'Features:     {FEATURES}')
print('\n✅ Scaler saved as feature_scaler.pkl')

## Cell 7 — Train Logistic Regression

In [ ]:
print('🔄 Training Logistic Regression...')

lr_model = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
lr_model.fit(X_train_scaled, y_train)

lr_train_pred = lr_model.predict(X_train_scaled)
lr_test_pred  = lr_model.predict(X_test_scaled)

lr_train_acc = accuracy_score(y_train, lr_train_pred)
lr_test_acc  = accuracy_score(y_test,  lr_test_pred)

print(f'\n✅ Logistic Regression Done!')
print(f'   Train Accuracy: {lr_train_acc:.4f} ({lr_train_acc*100:.2f}%)')
print(f'   Test  Accuracy: {lr_test_acc:.4f} ({lr_test_acc*100:.2f}%)')
print()
print('Classification Report (Test Set):')
print(classification_report(y_test, lr_test_pred,
      target_names=['No Event', 'Flood', 'Landslide']))

## Cell 8 — Train SVM

In [ ]:
print('🔄 Training SVM (this may take 2–5 mins)...')

svm_model = svm.SVC(kernel='linear', random_state=42, probability=True)
svm_model.fit(X_train_scaled, y_train)

svm_train_pred = svm_model.predict(X_train_scaled)
svm_test_pred  = svm_model.predict(X_test_scaled)

svm_train_acc = accuracy_score(y_train, svm_train_pred)
svm_test_acc  = accuracy_score(y_test,  svm_test_pred)

print(f'\n✅ SVM Done!')
print(f'   Train Accuracy: {svm_train_acc:.4f} ({svm_train_acc*100:.2f}%)')
print(f'   Test  Accuracy: {svm_test_acc:.4f} ({svm_test_acc*100:.2f}%)')
print()
print('Classification Report (Test Set):')
print(classification_report(y_test, svm_test_pred,
      target_names=['No Event', 'Flood', 'Landslide']))

## Cell 9 — Compare Both Models Visually

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison — Confusion Matrices (Test Set)', fontsize=14, fontweight='bold')

labels = ['No Event', 'Flood', 'Landslide']

# Logistic Regression
cm_lr = confusion_matrix(y_test, lr_test_pred)
ConfusionMatrixDisplay(cm_lr, display_labels=labels).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Logistic Regression\nTest Accuracy: {lr_test_acc*100:.2f}%')

# SVM
cm_svm = confusion_matrix(y_test, svm_test_pred)
ConfusionMatrixDisplay(cm_svm, display_labels=labels).plot(ax=axes[1], colorbar=False, cmap='Oranges')
axes[1].set_title(f'SVM\nTest Accuracy: {svm_test_acc*100:.2f}%')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

# Summary table
print('\n═══════════════════════════════════════')
print('         MODEL COMPARISON SUMMARY')
print('═══════════════════════════════════════')
print(f'  Logistic Regression → {lr_test_acc*100:.2f}%')
print(f'  SVM                 → {svm_test_acc*100:.2f}%')
print('═══════════════════════════════════════')
best = 'Logistic Regression' if lr_test_acc >= svm_test_acc else 'SVM'
print(f'  🏆 Winner: {best}')
print('═══════════════════════════════════════')

## Cell 10 — Save Best Model

In [ ]:
# Save whichever model performed better on test set
if lr_test_acc >= svm_test_acc:
    best_model = lr_model
    best_name  = 'Logistic Regression'
else:
    best_model = svm_model
    best_name  = 'SVM'

joblib.dump(best_model, 'nepal_disaster_model.pkl')

# Also save feature list so Streamlit knows the exact order
import json
with open('feature_columns.json', 'w') as f:
    json.dump(FEATURES, f)

# Save district list for the Streamlit dropdown
real_districts = sorted([d for d in df['district'].unique() if d != 'No Event'])
with open('district_list.json', 'w') as f:
    json.dump(real_districts, f)

print('Files saved for Streamlit app:')
print('  ✅ nepal_disaster_model.pkl    ← trained model')
print('  ✅ feature_scaler.pkl          ← StandardScaler')
print('  ✅ district_label_encoder.pkl  ← district → number')
print('  ✅ feature_columns.json        ← feature order')
print('  ✅ district_list.json          ← dropdown options')
print()
print(f'🏆 Best model: {best_name} ({max(lr_test_acc, svm_test_acc)*100:.2f}% accuracy)')
print()
print('🎉 Training complete! You are ready to build the Streamlit app.')

## Cell 11 — Test Prediction (Sanity Check)
Simulate what the Streamlit app will do — give it real-looking inputs and see what it predicts.

In [ ]:
label_map = {0: '🟢 No Event', 1: '🔴 Flood Risk', 2: '🟠 Landslide Risk'}

def predict_disaster(district_name, precipitation, rainfall_3day,
                     rainfall_7day, rainfall_30day, temperature,
                     temp_7day, month):
    is_monsoon = 1 if 6 <= month <= 9 else 0
    season = 2 if 6 <= month <= 9 else (1 if 3 <= month <= 5 else (3 if month in [10,11] else 0))
    district_enc = le.transform([district_name])[0]

    input_data = np.array([[precipitation, rainfall_3day, rainfall_7day,
                            rainfall_30day, temperature, temp_7day,
                            month, is_monsoon, season, district_enc]])
    input_scaled = scaler.transform(input_data)
    pred = best_model.predict(input_scaled)[0]
    proba = best_model.predict_proba(input_scaled)[0]

    print(f'District:      {district_name}')
    print(f'Month:         {month} | Monsoon: {"Yes" if is_monsoon else "No"}')
    print(f'Precipitation: {precipitation} mm')
    print(f'3-day total:   {rainfall_3day} mm')
    print(f'7-day total:   {rainfall_7day} mm')
    print()
    print(f'Prediction → {label_map[pred]}')
    print(f'Confidence:   No Event={proba[0]*100:.1f}%  Flood={proba[1]*100:.1f}%  Landslide={proba[2]*100:.1f}%')
    print('─' * 50)

# Test 1: Heavy monsoon rain in Sindhupalchok (landslide-prone)
print('TEST 1 — Heavy monsoon rain in Sindhupalchok')
predict_disaster('sindhupalchok', precipitation=85, rainfall_3day=210,
                 rainfall_7day=390, rainfall_30day=620, temperature=22,
                 temp_7day=21, month=8)

# Test 2: Dry winter day in Kathmandu
print('TEST 2 — Dry winter day in Kathmandu')
predict_disaster('kathmandu', precipitation=0, rainfall_3day=2,
                 rainfall_7day=5, rainfall_30day=18, temperature=12,
                 temp_7day=13, month=1)

# Test 3: Moderate rain in Terai flood zone
print('TEST 3 — Moderate rain in Morang (Terai flood zone)')
predict_disaster('morang', precipitation=55, rainfall_3day=140,
                 rainfall_7day=260, rainfall_30day=450, temperature=28,
                 temp_7day=27, month=7)